## Lake Sarez Landslides, Tajikistan (2017)

This exploration reproduces the findings shared in the following paper: [Integration of satellite SAR and optical acquisitions for the characterization of the Lake Sarez landslides in Tajikistan](https://www.researchgate.net/publication/378176884_Integration_of_satellite_SAR_and_optical_acquisitions_for_the_characterization_of_the_Lake_Sarez_landslides_in_Tajikistan).

The PyGMTSAR InSAR library, Geomed3D Geophysical Inversion Library, N-Cube 3D/4D GIS Data Visualization, among others, are my open-source projects developed in my free time. I hold a Master's degree in STEM, specializing in radio physics. In 2004, I received the first prize in the All-Russian Physics Competition for significant results in forward and inverse modeling for nonlinear optics and holography. These skills are also applicable to modeling Gravity, Magnetic, and Thermal fields, as well as satellite interferometry processing. With 20 years of experience as a data scientist and software developer, I have contributed to scientific and industrial development, working on government contracts, university projects, and with companies like LG Corp and Google Inc.

You can support my work on [Patreon](https://www.patreon.com/pechnikov), where I share updates on my projects, publications, use cases, examples, and other useful information. For research and development services and support, please visit my profile on the freelance platform [Upwork](https://www.upwork.com).

### Resources
- Google Colab Pro notebooks and articles on [Patreon](https://www.patreon.com/pechnikov),
- Google Colab notebooks on [GitHub](https://github.com),
- Docker Images on [DockerHub](https://hub.docker.com),
- Geological Models on [YouTube](https://www.youtube.com),
- VR/AR Geological Models on [GitHub](https://github.com),
- Live updates and announcements on [LinkedIn](https://www.linkedin.com/in/alexey-pechnikov/).

© Alexey Pechnikov, 2024

$\large\color{blue}{\text{Hint: Use menu Cell} \to \text{Run All or Runtime} \to \text{Complete All or Runtime} \to \text{Run All}}$
$\large\color{blue}{\text{(depending of your localization settings) to execute the entire notebook}}$

## Google Colab Installation

Install PyGMTSAR and required GMTSAR binaries (including SNAPHU)

In [19]:
# --- Set notebook name manually (recommended workaround) ---
notebook_name = "Fill06_InSAR"   # 👈 replace with your notebook's actual name
print(f"✅ Notebook name set as: {notebook_name}")


✅ Notebook name set as: Fill06_InSAR


In [20]:
import platform, sys, os
if 'google.colab' in sys.modules:
    # install PyGMTSAR stable version from PyPI
    !{sys.executable} -m pip install -q pygmtsar
    # alternatively, nstall PyGMTSAR development version from GitHub
    #!{sys.executable} -m pip install -Uq git+https://github.com/mobigroup/gmtsar.git@pygmtsar2#subdirectory=pygmtsar
    # use PyGMTSAR Google Colab installation script to install binary dependencies
    # script URL: https://github.com/AlexeyPechnikov/pygmtsar/blob/pygmtsar2/pygmtsar/pygmtsar/data/google_colab.sh
    import importlib.resources as resources
    with resources.as_file(resources.files('pygmtsar.data') / 'google_colab.sh') as google_colab_script_filename:
        !sh {google_colab_script_filename}
    # enable custom widget manager as required by recent Google Colab updates
    from google.colab import output
    output.enable_custom_widget_manager()
    # initialize virtual framebuffer for interactive 3D visualization; required for headless environments
    import xvfbwrapper
    display = xvfbwrapper.Xvfb(width=800, height=600)
    display.start()

# specify GMTSAR installation path
PATH = os.environ['PATH']
if PATH.find('GMTSAR') == -1:
    PATH = os.environ['PATH'] + ':/usr/local/GMTSAR/bin/'
    %env PATH {PATH}

# display PyGMTSAR version
from pygmtsar import __version__
__version__

'2025.4.8.post1'

## Load and Setup Python Modules

In [21]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import json
from dask.distributed import Client
import dask
import warnings
warnings.filterwarnings('ignore')

In [22]:
# plotting modules
import pyvista as pv
# magic trick for white background
pv.set_plot_theme("document")
import panel
panel.extension(comms='ipywidgets')
panel.extension('vtk')
from contextlib import contextmanager
import matplotlib.pyplot as plt
@contextmanager
def mpl_settings(settings):
    original_settings = {k: plt.rcParams[k] for k in settings}
    plt.rcParams.update(settings)
    yield
    plt.rcParams.update(original_settings)
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.titlesize'] = 24
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
%matplotlib inline

In [23]:
# define Pandas display settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

In [24]:
from pygmtsar import S1, Stack, tqdm_dask, ASF, Tiles, XYZTiles, utils

# recent Google Colab changes in early September 2025 broke Dask+Xarray NetCDF multithedded processing (again)
# workaround below disables multitheading when it does not work, degrading performance and increasing RAM usage.
if 'google.colab' in sys.modules:
    methods = {
        "load_dem":  "synchronous",
        "save_cube": "compute",
        "save_stack":"compute",
    }
    for m, kind in methods.items():
        if not hasattr(Stack, f"_{m}"):
            setattr(Stack, f"_{m}", getattr(Stack, m))
        def _make_wrapper(name, kind):
            orig = getattr(Stack, f"_{name}")
            if kind == "synchronous":
                def _wrapper(self, *args, **kwargs):
                    with dask.config.set(scheduler="synchronous"):
                        return orig(self, *args, **kwargs)
                return _wrapper
            elif kind == "compute":
                def _wrapper(self, *args, **kwargs):
                    if args:
                        return orig(self, args[0].compute() if hasattr(args[0], "compute") else args[0], *args[1:], **kwargs)
                    return orig(self, **kwargs)
                return _wrapper
            else:
                raise NotImplementedError(f"Unknown wrapper kind: {kind}")
        setattr(Stack, m, _make_wrapper(m, kind))


In [25]:
import sys
import time
from IPython.display import Javascript, display

# ✅ Auto-allow downloads function
def enable_auto_downloads():
    """Enable automatic downloads in Colab"""
    if 'google.colab' in sys.modules:
        # JavaScript to handle download permissions automatically
        js_code = """
        // Auto-allow downloads
        window.addEventListener('beforeunload', function() {
            return null;
        });

        // Override download permission
        if (window.navigator && window.navigator.permissions) {
            window.navigator.permissions.query({name: 'downloads'}).then(function(result) {
                if (result.state === 'granted') {
                    console.log('Download permission already granted');
                } else {
                    console.log('Download permission requested');
                }
            });
        }

        // Allow multiple downloads
        window.open = function(url, target, features) {
            const newWindow = window.parent.open(url, target, features);
            return newWindow;
        };

        console.log('✅ Auto-download permissions enabled');
        """
        display(Javascript(js_code))
        time.sleep(1)
        print("✅ Auto-download permissions enabled!")

# Call at the beginning of your notebook
enable_auto_downloads()


<IPython.core.display.Javascript object>

✅ Auto-download permissions enabled!


## Define Sentinel-1 SLC Scenes and Processing Parameters

### Descending Orbit Configuration

https://search.asf.alaska.edu/#/?polygon=POINT(72.66%2038.25)&start=2017-05-11T00:00:01Z&end=2017-12-25T23:59:59Z&productTypes=SLC&resultsLoaded=true&zoom=7.131&center=74.457,35.638&path=5-5&frame=466-466

In [26]:
# The subswath is required for partial scene downloads and is not used for burst downloads.
# The orbit is used to define directory names.
ORBIT = 'D'
SUBSWATH = 3
REFERENCE = '2022-06-29'

In [27]:
# SCENES = """
# S1A_IW_SLC__1SDV_20241221T005138_20241221T005206_057083_070448_762F
# S1A_IW_SLC__1SDV_20241022T005142_20241022T005209_056208_06E17F_C479
# S1A_IW_SLC__1SDV_20230501T005138_20230501T005205_048333_05D01F_CE49
# S1A_IW_SLC__1SDV_20230218T005136_20230218T005204_047283_05ACB0_401C
# S1A_IW_SLC__1SDV_20220810T005138_20220810T005206_044483_054EEC_10FE
# S1A_IW_SLC__1SDV_20220705T005132_20220705T005200_043958_053F54_8668
# S1A_IW_SLC__1SDV_20211107T005130_20211107T005158_040458_04CC07_3165
# S1A_IW_SLC__1SDV_20210920T005130_20210920T005158_039758_04B3BB_6A09
# S1A_IW_SLC__1SDV_20201206T005124_20201206T005152_035558_042875_B92B
# S1A_IW_SLC__1SDV_20200105T005118_20200105T005145_030658_03837F_FEFD
# # """
# SCENES = list(filter(None, SCENES.split('\n')))
# print (f'Scenes defined: {len(SCENES)}')

In [28]:
import asf_search as asf

# Define a WKT point forthe area of interest
wkt = 'POINT(88.526 27.484)'

# Define the start and end dates for the search
start_date = '2020-01-01'
end_date = '2024-12-30'

# Perform the geo-search with filtering for Sentinel-1A, Burst processing level
results = asf.geo_search(
    platform=[asf.PLATFORM.SENTINEL1A],  # Only Sentinel-1A
    processingLevel=asf.PRODUCT_TYPE.BURST,
    intersectsWith=wkt,
    start=start_date,
    end=end_date,
    polarization=[asf.POLARIZATION.VV],  # VV only; change to [asf.POLARIZATION.VV, asf.POLARIZATION.VH] for both
    flightDirection=asf.FLIGHT_DIRECTION.DESCENDING  # Remove this line if you want both ascending and descending paths
)

# Extract scene names
BURSTS = [result.properties.get('sceneName', 'N/A') for result in results]

# Print all burst scene names
for burst in BURSTS:
    print(f"# {burst}")

# Print total count of bursts
print(f'Total Bursts Found: {len(BURSTS)}')

# S1_101865_IW2_20241227T000337_VV_97C6-BURST
# S1_101865_IW2_20241215T000338_VV_25EA-BURST
# S1_101865_IW2_20241203T000339_VV_C312-BURST
# S1_101865_IW2_20241121T000340_VV_3CE4-BURST
# S1_101865_IW2_20241109T000341_VV_C388-BURST
# S1_101865_IW2_20241028T000341_VV_082D-BURST
# S1_101865_IW2_20241016T000341_VV_1FB7-BURST
# S1_101865_IW2_20241004T000341_VV_FA07-BURST
# S1_101865_IW2_20240922T000340_VV_3F9A-BURST
# S1_101865_IW2_20240910T000340_VV_33CF-BURST
# S1_101865_IW2_20240829T000340_VV_5060-BURST
# S1_101865_IW2_20240817T000339_VV_2D0E-BURST
# S1_101865_IW2_20240805T000340_VV_3F4A-BURST
# S1_101865_IW2_20240724T000340_VV_74BA-BURST
# S1_101865_IW2_20240712T000340_VV_0600-BURST
# S1_101865_IW2_20240618T000341_VV_5C57-BURST
# S1_101865_IW2_20240606T000341_VV_4EAA-BURST
# S1_101865_IW2_20240525T000342_VV_3E42-BURST
# S1_101865_IW2_20240501T000342_VV_A3DB-BURST
# S1_101865_IW2_20240419T000342_VV_BE92-BURST
# S1_101865_IW2_20240407T000342_VV_0E7F-BURST
# S1_101865_IW2_20240326T000341_VV

https://search.asf.alaska.edu/#/?polygon=POLYGON((72.72%2038.26,72.6936%2038.3236,72.63%2038.35,72.5664%2038.3236,72.54%2038.26,72.5664%2038.1964,72.63%2038.17,72.6936%2038.1964,72.72%2038.26))&start=2017-05-11T00:00:01Z&end=2017-12-25T23:59:59Z&resultsLoaded=true&zoom=10.025&center=72.469,38.068&path=5-5&frame=466-466&granule=S1_009440_IW2_20171225T011414_VV_4328-BURST&dataset=SENTINEL-1%20BURSTS&polarizations=VV

In [29]:
# BURSTS = """
# S1_134075_IW1_20200106T004321_VH_79A9-BURST
# S1_134075_IW1_20200106T004321_VV_79A9-BURST
# S1_134075_IW1_20200112T004402_VH_E857-BURST
# S1_134075_IW1_20200112T004402_VV_E857-BURST
# S1_134075_IW1_20200118T004320_VH_E8C5-BURST
# S1_134075_IW1_20200118T004320_VV_E8C5-BURST
# S1_134075_IW1_20200124T004402_VH_796C-BURST
# S1_134075_IW1_20200124T004402_VV_796C-BURST
# S1_134075_IW1_20200312T004401_VH_2B0B-BURST
# S1_134075_IW1_20200312T004401_VV_2B0B-BURST
# S1_134075_IW1_20200324T004401_VH_6B6E-BURST
# S1_134075_IW1_20200324T004401_VV_6B6E-BURST
# S1_134075_IW1_20200405T004402_VH_D7BA-BURST
# S1_134075_IW1_20200405T004402_VV_D7BA-BURST
# S1_134075_IW1_20200405T004402_VV_D7BA-BURST
# S1_134075_IW1_20200417T004402_VH_6E84-BURST
# S1_134075_IW1_20200417T004402_VV_6E84-BURST
# S1_134075_IW1_20200429T004403_VH_4F7F-BURST
# S1_134075_IW1_20200429T004403_VV_4F7F-BURST
# S1_134075_IW1_20200511T004403_VH_0316-BURST
# S1_134075_IW1_20200511T004403_VV_0316-BURST
# S1_134075_IW1_20200523T004404_VH_FD8C-BURST
# S1_134075_IW1_20200523T004404_VV_FD8C-BURST
# S1_134075_IW1_20200604T004405_VH_FE0E-BURST
# S1_134075_IW1_20200604T004405_VV_FE0E-BURST
# S1_134075_IW1_20200616T004406_VV_4420-BURST
# S1_134075_IW1_20200628T004406_VH_63B1-BURST
# S1_134075_IW1_20200628T004406_VV_63B1-BURST
# S1_134075_IW1_20200628T004406_VV_63B1-BURST
# S1_134075_IW1_20200710T004407_VV_6582-BURST
# S1_134075_IW1_20200722T004408_VH_E789-BURST
# S1_134075_IW1_20200722T004408_VV_E789-BURST
# S1_134075_IW1_20200803T004408_VH_443C-BURST
# S1_134075_IW1_20200803T004408_VV_443C-BURST
# S1_134075_IW1_20200815T004409_VH_7B1A-BURST
# S1_134075_IW1_20200815T004409_VV_7B1A-BURST
# S1_134075_IW1_20200827T004410_VH_7EBE-BURST
# S1_134075_IW1_20200827T004410_VV_7EBE-BURST
# S1_134075_IW1_20200908T004410_VH_5F55-BURST
# S1_134075_IW1_20200908T004410_VV_5F55-BURST
# S1_134075_IW1_20200920T004411_VH_CEF9-BURST
# S1_134075_IW1_20200920T004411_VV_CEF9-BURST
# S1_134075_IW1_20201002T004411_VH_94B6-BURST
# S1_134075_IW1_20201002T004411_VV_94B6-BURST
# S1_134075_IW1_20201014T004411_VH_0CD0-BURST
# S1_134075_IW1_20201014T004411_VV_0CD0-BURST
# S1_134075_IW1_20201026T004411_VH_05BE-BURST
# S1_134075_IW1_20201107T004411_VV_DA3A-BURST
# S1_134075_IW1_20201119T004411_VH_0B86-BURST
# S1_134075_IW1_20201119T004411_VV_0B86-BURST
# S1_134075_IW1_20201201T004410_VH_F38D-BURST
# S1_134075_IW1_20201201T004410_VV_F38D-BURST
# S1_134075_IW1_20201213T004410_VH_1446-BURST
# S1_134075_IW1_20201213T004410_VV_1446-BURST
# S1_134075_IW1_20201225T004409_VH_A1DB-BURST
# S1_134075_IW1_20201225T004409_VV_A1DB-BURST
# S1_134075_IW1_20210106T004409_VH_A822-BURST
# S1_134075_IW1_20210106T004409_VV_A822-BURST
# S1_134075_IW1_20210118T004408_VH_E263-BURST
# S1_134075_IW1_20210118T004408_VV_E263-BURST
# S1_134075_IW1_20210130T004408_VH_D908-BURST
# S1_134075_IW1_20210130T004408_VV_D908-BURST
# S1_134075_IW1_20210211T004407_VH_B3FE-BURST
# S1_134075_IW1_20210211T004407_VV_B3FE-BURST
# S1_134075_IW1_20210217T004325_VH_570E-BURST
# S1_134075_IW1_20210217T004325_VV_570E-BURST
# S1_134075_IW1_20210223T004407_VH_57A0-BURST
# S1_134075_IW1_20210307T004407_VH_B2B1-BURST
# S1_134075_IW1_20210307T004407_VV_B2B1-BURST
# S1_134075_IW1_20210307T004407_VV_B2B1-BURST
# S1_134075_IW1_20210319T004407_VH_8BCE-BURST
# S1_134075_IW1_20210319T004407_VV_8BCE-BURST
# S1_134075_IW1_20210331T004407_VH_CFCB-BURST
# S1_134075_IW1_20210331T004407_VH_CFCB-BURST
# S1_134075_IW1_20210412T004408_VH_EB7E-BURST
# S1_134075_IW1_20210412T004408_VV_EB7E-BURST
# S1_134075_IW1_20210424T004408_VH_6B9E-BURST
# S1_134075_IW1_20210424T004408_VV_6B9E-BURST
# S1_134075_IW1_20210506T004409_VH_4A42-BURST
# S1_134075_IW1_20210506T004409_VV_4A42-BURST
# S1_134075_IW1_20210518T004409_VH_27A6-BURST
# S1_134075_IW1_20210518T004409_VV_27A6-BURST
# S1_134075_IW1_20210530T004410_VH_4572-BURST
# S1_134075_IW1_20210530T004410_VV_4572-BURST
# S1_134075_IW1_20210530T004410_VV_4572-BURST
# S1_134075_IW1_20210623T004412_VH_4E05-BURST
# S1_134075_IW1_20210623T004412_VV_4E05-BURST
# S1_134075_IW1_20210623T004412_VV_4E05-BURST
# S1_134075_IW1_20210705T004412_VV_CE29-BURST
# S1_134075_IW1_20210717T004413_VH_7C6D-BURST
# S1_134075_IW1_20210729T004414_VH_2C8E-BURST
# S1_134075_IW1_20210729T004414_VV_2C8E-BURST
# S1_134075_IW1_20210810T004415_VH_BE10-BURST
# S1_134075_IW1_20210810T004415_VV_BE10-BURST
# S1_134075_IW1_20210822T004415_VH_C04A-BURST
# S1_134075_IW1_20210822T004415_VV_C04A-BURST
# S1_134075_IW1_20210903T004416_VH_A1EB-BURST
# S1_134075_IW1_20210903T004416_VV_A1EB-BURST
# S1_134075_IW1_20210915T004416_VH_FCC5-BURST
# S1_134075_IW1_20210915T004416_VV_FCC5-BURST
# S1_134075_IW1_20210927T004417_VH_519E-BURST
# S1_134075_IW1_20210927T004417_VV_519E-BURST
# S1_134075_IW1_20211009T004417_VH_DD3D-BURST
# S1_134075_IW1_20211009T004417_VV_DD3D-BURST
# S1_134075_IW1_20211021T004417_VH_4B8A-BURST
# S1_134075_IW1_20211021T004417_VH_4B8A-BURST
# S1_134075_IW1_20211021T004417_VV_4B8A-BURST
# S1_134075_IW1_20211102T004417_VH_DEF5-BURST
# S1_134075_IW1_20211102T004417_VV_DEF5-BURST
# S1_134075_IW1_20211114T004417_VH_2D9B-BURST
# S1_134075_IW1_20211114T004417_VV_2D9B-BURST
# S1_134075_IW1_20211126T004416_VH_B8A8-BURST
# S1_134075_IW1_20211126T004416_VV_B8A8-BURST
# S1_134075_IW1_20211208T004416_VH_9F5E-BURST
# S1_134075_IW1_20211208T004416_VV_9F5E-BURST
# S1_134075_IW1_20211220T004415_VH_59A0-BURST
# S1_134075_IW1_20211220T004415_VV_59A0-BURST
# S1_134075_IW1_20220101T004414_VH_AED5-BURST
# S1_134075_IW1_20220101T004414_VV_AED5-BURST
# S1_134075_IW1_20220113T004414_VV_C45D-BURST
# S1_134075_IW1_20220206T004413_VH_A3AF-BURST
# S1_134075_IW1_20220206T004413_VV_A3AF-BURST
# S1_134075_IW1_20220218T004413_VH_7F8A-BURST
# S1_134075_IW1_20220314T004413_VH_38D3-BURST
# S1_134075_IW1_20220314T004413_VV_38D3-BURST
# S1_134075_IW1_20220326T004413_VH_DBF9-BURST
# S1_134075_IW1_20220407T004413_VH_C2C6-BURST
# S1_134075_IW1_20220407T004413_VV_C2C6-BURST
# S1_134075_IW1_20220419T004414_VH_1966-BURST
# S1_134075_IW1_20220419T004414_VV_1966-BURST
# S1_134075_IW1_20220501T004414_VH_AF66-BURST
# S1_134075_IW1_20220501T004414_VV_AF66-BURST
# S1_134075_IW1_20220513T004415_VH_74A5-BURST
# S1_134075_IW1_20220513T004415_VV_74A5-BURST
# S1_134075_IW1_20220525T004416_VH_ACAD-BURST
# S1_134075_IW1_20220525T004416_VV_ACAD-BURST
# S1_134075_IW1_20220606T004417_VH_9A06-BURST
# S1_134075_IW1_20220606T004417_VV_9A06-BURST
# S1_134075_IW1_20220618T004418_VH_F5DA-BURST
# S1_134075_IW1_20220618T004418_VV_F5DA-BURST
# S1_134075_IW1_20220630T004418_VH_6E84-BURST
# S1_134075_IW1_20220712T004419_VH_2080-BURST
# S1_134075_IW1_20220712T004419_VV_2080-BURST
# S1_134075_IW1_20220724T004420_VH_A2E4-BURST
# S1_134075_IW1_20220805T004421_VV_912F-BURST
# S1_134075_IW1_20220817T004421_VH_821C-BURST
# S1_134075_IW1_20220817T004421_VV_821C-BURST
# S1_134075_IW1_20220829T004422_VH_1CBD-BURST
# S1_134075_IW1_20220829T004422_VV_1CBD-BURST
# S1_134075_IW1_20220829T004422_VV_1CBD-BURST
# S1_134075_IW1_20220910T004423_VH_465A-BURST
# S1_134075_IW1_20220910T004423_VV_465A-BURST
# S1_134075_IW1_20220910T004423_VV_465A-BURST
# S1_134075_IW1_20220922T004422_VH_7774-BURST
# S1_134075_IW1_20220922T004422_VV_7774-BURST
# S1_134075_IW1_20221004T004423_VH_5089-BURST
# S1_134075_IW1_20221004T004423_VV_5089-BURST
# S1_134075_IW1_20221016T004423_VH_7B83-BURST
# S1_134075_IW1_20221016T004423_VV_7B83-BURST
# S1_134075_IW1_20221028T004423_VH_BCC1-BURST
# S1_134075_IW1_20221028T004423_VH_BCC1-BURST
# S1_134075_IW1_20221109T004422_VH_B6D5-BURST
# S1_134075_IW1_20221109T004422_VV_B6D5-BURST
# S1_134075_IW1_20221109T004422_VV_B6D5-BURST
# S1_134075_IW1_20221121T004423_VH_E38D-BURST
# S1_134075_IW1_20221121T004423_VV_E38D-BURST
# S1_134075_IW1_20221203T004422_VH_5149-BURST
# S1_134075_IW1_20221203T004422_VV_5149-BURST
# S1_134075_IW1_20221215T004421_VV_A101-BURST
# S1_134075_IW1_20221227T004420_VH_5DC0-BURST
# S1_134075_IW1_20221227T004420_VV_5DC0-BURST
# S1_134075_IW1_20230108T004420_VH_FFD6-BURST
# S1_134075_IW1_20230108T004420_VV_FFD6-BURST
# S1_134075_IW1_20230201T004419_VH_8707-BURST
# S1_134075_IW1_20230201T004419_VV_8707-BURST
# S1_134075_IW1_20230213T004419_VH_E945-BURST
# S1_134075_IW1_20230213T004419_VV_E945-BURST
# S1_134075_IW1_20230225T004418_VH_B3D2-BURST
# S1_134075_IW1_20230225T004418_VV_B3D2-BURST
# S1_134075_IW1_20230426T004420_VH_2BBA-BURST
# S1_134075_IW1_20230426T004420_VV_2BBA-BURST
# S1_134075_IW1_20230520T004421_VH_8FB4-BURST
# S1_134075_IW1_20230520T004421_VV_8FB4-BURST
# S1_134075_IW1_20230601T004422_VH_0FDA-BURST
# S1_134075_IW1_20230601T004422_VV_0FDA-BURST
# S1_134075_IW1_20230613T004422_VH_0607-BURST
# S1_134075_IW1_20230613T004422_VV_0607-BURST
# S1_134075_IW1_20230625T004423_VH_2D1A-BURST
# S1_134075_IW1_20230625T004423_VV_2D1A-BURST
# S1_134075_IW1_20230707T004424_VH_087B-BURST
# S1_134075_IW1_20230707T004424_VV_087B-BURST
# S1_134075_IW1_20230719T004424_VV_BB00-BURST
# S1_134075_IW1_20230731T004425_VH_8D4C-BURST
# S1_134075_IW1_20230731T004425_VV_8D4C-BURST
# S1_134075_IW1_20230812T004425_VH_2F2F-BURST
# S1_134075_IW1_20230812T004425_VV_2F2F-BURST
# S1_134075_IW1_20230824T004426_VH_92E9-BURST
# S1_134075_IW1_20230905T004427_VH_61EE-BURST
# S1_134075_IW1_20230905T004427_VV_61EE-BURST
# S1_134075_IW1_20230917T004428_VH_CF1F-BURST
# S1_134075_IW1_20230917T004428_VV_CF1F-BURST
# S1_134075_IW1_20231011T004427_VH_0837-BURST
# S1_134075_IW1_20231011T004427_VV_0837-BURST
# S1_134075_IW1_20231023T004428_VH_ABDF-BURST
# S1_134075_IW1_20231023T004428_VV_ABDF-BURST
# S1_134075_IW1_20231104T004427_VH_8554-BURST
# S1_134075_IW1_20231104T004427_VV_8554-BURST
# S1_134075_IW1_20231116T004427_VH_9144-BURST
# S1_134075_IW1_20231116T004427_VV_9144-BURST
# S1_134075_IW1_20241228T004420_VV_6AE8-BURST
# S1_134075_IW1_20241228T004420_VH_6AE8-BURST
# S1_134075_IW1_20241216T004421_VV_8C25-BURST
# S1_134075_IW1_20241216T004421_VV_8C25-BURST
# S1_134075_IW1_20241216T004421_VH_8C25-BURST
# S1_134075_IW1_20241204T004422_VV_CAC8-BURST
# S1_134075_IW1_20241204T004422_VH_CAC8-BURST
# S1_134075_IW1_20241122T004423_VV_24BB-BURST
# S1_134075_IW1_20241122T004423_VH_24BB-BURST
# S1_134075_IW1_20241122T004423_VH_24BB-BURST
# S1_134075_IW1_20241110T004424_VV_25C9-BURST
# S1_134075_IW1_20241110T004424_VH_25C9-BURST
# S1_134075_IW1_20241029T004424_VV_447B-BURST
# S1_134075_IW1_20241029T004424_VH_447B-BURST
# S1_134075_IW1_20241005T004424_VV_FF85-BURST
# S1_134075_IW1_20241005T004424_VH_FF85-BURST
# S1_134075_IW1_20240923T004424_VV_9234-BURST
# S1_134075_IW1_20240923T004424_VH_9234-BURST
# S1_134075_IW1_20240911T004423_VV_A434-BURST
# S1_134075_IW1_20240911T004423_VH_A434-BURST
# S1_134075_IW1_20240830T004423_VV_CDBD-BURST
# S1_134075_IW1_20240830T004423_VH_CDBD-BURST
# S1_134075_IW1_20240818T004423_VV_E89B-BURST
# S1_134075_IW1_20240818T004423_VH_E89B-BURST
# S1_134075_IW1_20240806T004423_VH_D0E9-BURST
# S1_134075_IW1_20240725T004422_VV_7150-BURST
# S1_134075_IW1_20240725T004422_VH_7150-BURST
# S1_134075_IW1_20240713T004423_VV_F3EA-BURST
# S1_134075_IW1_20240713T004423_VH_F3EA-BURST
# S1_134075_IW1_20240701T004423_VV_DF8E-BURST
# S1_134075_IW1_20240701T004423_VH_DF8E-BURST
# S1_134075_IW1_20240619T004424_VV_7F06-BURST
# S1_134075_IW1_20240607T004424_VH_E1D5-BURST
# S1_134075_IW1_20240526T004425_VV_4135-BURST
# S1_134075_IW1_20240526T004425_VH_4135-BURST
# S1_134075_IW1_20240514T004425_VV_0B45-BURST
# S1_134075_IW1_20240514T004425_VH_0B45-BURST
# S1_134075_IW1_20240502T004425_VV_C799-BURST
# S1_134075_IW1_20240502T004425_VH_C799-BURST
# S1_134075_IW1_20240420T004425_VV_3A2C-BURST
# S1_134075_IW1_20240420T004425_VH_3A2C-BURST
# S1_134075_IW1_20240408T004425_VH_8085-BURST
# S1_134075_IW1_20240327T004424_VV_1239-BURST
# S1_134075_IW1_20240327T004424_VH_1239-BURST
# S1_134075_IW1_20240315T004424_VV_B0F9-BURST
# S1_134075_IW1_20240315T004424_VH_B0F9-BURST
# S1_134075_IW1_20240220T004424_VV_A17B-BURST
# S1_134075_IW1_20240220T004424_VH_A17B-BURST
# S1_134075_IW1_20240208T004424_VV_D552-BURST
# S1_134075_IW1_20240208T004424_VH_D552-BURST
# S1_134075_IW1_20240208T004424_VH_D552-BURST
# S1_134075_IW1_20240127T004424_VH_4C29-BURST
# S1_134075_IW1_20231116T004427_VV_9144-BURST
# S1_134075_IW1_20231116T004427_VH_9144-BURST
# S1_134075_IW1_20231104T004427_VV_8554-BURST
# S1_134075_IW1_20231104T004427_VH_8554-BURST
# """
# BURSTS = list(filter(None, BURSTS.split('\n')))
# print (f'Bursts defined: {len(BURSTS)}')

In [30]:
WORKDIR = 'raw_sarez2017_' + 'desc'  if ORBIT == 'D' else 'asc'
DATADIR = 'data_sarez2017_' + 'desc' if ORBIT == 'D' else 'asc'

In [31]:
# define DEM filename inside data directory
DEM = f'{DATADIR}/dem.nc'

In [32]:
# subsidence point from
geojson = '''
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [88.526, 27.484]
  },
  "properties": {}
}
'''
POI = gpd.GeoDataFrame.from_features([json.loads(geojson)])
POI

,geometry
0,POINT (88.526 27.484)


In [33]:
geojson = '''
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [88.526, 27.484]
  },
  "properties": {}
}
'''
BUFFER = 0.06
AOI = gpd.GeoDataFrame.from_features([json.loads(geojson)])
AOI['geometry'] = AOI.buffer(BUFFER)
AOI

,geometry
0,"POLYGON ((88.586 27.484, 88.58571 27.47812, 88.58485 27.47229, 88.58342 27.46658, 88.58143 27.46..."


## Download and Unpack Datasets

## Enter Your ASF User and Password

If the data directory is empty or doesn't exist, you'll need to download Sentinel-1 scenes from the Alaska Satellite Facility (ASF) datastore. Use your Earthdata Login credentials. If you don't have an Earthdata Login, you can create one at https://urs.earthdata.nasa.gov//users/new

You can also use pre-existing SLC scenes stored on your Google Drive, or you can copy them using a direct public link from iCloud Drive.

The credentials below are available at the time the notebook is validated.

In [34]:
# Set these variables to None and you will be prompted to enter your username and password below.
asf_username = 'GoogleColab2023'
asf_password = 'GoogleColab_2023'

In [35]:
# Set these variables to None and you will be prompted to enter your username and password below.
asf = ASF(asf_username, asf_password)
# Optimized scene downloading from ASF - only the required subswaths and polarizations.
# Subswaths are already encoded in burst identifiers and are only needed for scenes.
#print(asf.download(DATADIR, SCENES, SUBSWATH))
print(asf.download(DATADIR, BURSTS))

ASF Downloading Bursts Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ASF Downloading Sentinel-1 SLC Bursts:   0%|          | 0/145 [00:00<?, ?it/s]

                                  burst_or_scene
0    S1_101865_IW2_20241227T000337_VV_97C6-BURST
1    S1_101865_IW2_20241215T000338_VV_25EA-BURST
2    S1_101865_IW2_20241203T000339_VV_C312-BURST
3    S1_101865_IW2_20241121T000340_VV_3CE4-BURST
4    S1_101865_IW2_20241109T000341_VV_C388-BURST
5    S1_101865_IW2_20241028T000341_VV_082D-BURST
6    S1_101865_IW2_20241016T000341_VV_1FB7-BURST
7    S1_101865_IW2_20241004T000341_VV_FA07-BURST
8    S1_101865_IW2_20240922T000340_VV_3F9A-BURST
9    S1_101865_IW2_20240910T000340_VV_33CF-BURST
10   S1_101865_IW2_20240829T000340_VV_5060-BURST
11   S1_101865_IW2_20240817T000339_VV_2D0E-BURST
12   S1_101865_IW2_20240805T000340_VV_3F4A-BURST
13   S1_101865_IW2_20240724T000340_VV_74BA-BURST
14   S1_101865_IW2_20240712T000340_VV_0600-BURST
15   S1_101865_IW2_20240618T000341_VV_5C57-BURST
16   S1_101865_IW2_20240606T000341_VV_4EAA-BURST
17   S1_101865_IW2_20240525T000342_VV_3E42-BURST
18   S1_101865_IW2_20240501T000342_VV_A3DB-BURST
19   S1_101865_IW2_2

In [ ]:
# scan the data directory for SLC scenes and download missed orbits
S1.download_orbits(DATADIR, S1.scan_slc(DATADIR,subswath=2))

In [ ]:
# download Copernicus Global DEM 1 arc-second
Tiles().download_dem(AOI, filename=DEM).plot.imshow(cmap='cividis')

## Run Local Dask Cluster

Launch Dask cluster for local and distributed multicore computing. That's possible to process terabyte scale Sentinel-1 SLC datasets on Apple Air 16 GB RAM.

In [ ]:
# simple Dask initialization
if 'client' in globals():
    client.close()
client = Client()
client

## Init

Search recursively for measurement (.tiff) and annotation (.xml) and orbit (.EOF) files in the DATA directory. It can be directory with full unzipped scenes (.SAFE) subdirectories or just a directory with the list of pairs of required .tiff and .xml files (maybe pre-filtered for orbit, polarization and subswath to save disk space). If orbit files and DEM are missed these will be downloaded automatically below.

### Select Original Secenes and Orbits

Use filters to find required subswath, polarization and orbit in original scenes .SAFE directories in the data directory.

In [ ]:
scenes = S1.scan_slc(DATADIR,subswath=2)

In [ ]:
sbas = Stack(WORKDIR, drop_if_exists=True).set_scenes(scenes).set_reference(REFERENCE)
sbas.to_dataframe()

In [ ]:
sbas.plot_scenes(AOI=AOI)

## Reframe Scenes (Optional)

Stitch sequential scenes and crop the subswath to a smaller area for faster processing when the full area is not needed.

In [ ]:
sbas.compute_reframe(AOI)

In [ ]:
sbas.plot_scenes(AOI=AOI)

### Load DEM

The function below loads DEM from file or Xarray variable and converts heights to ellipsoidal model using EGM96 grid.

In [ ]:
sbas.load_dem(DEM, AOI)

In [ ]:
sbas.plot_scenes(AOI=AOI)

## Align Images

In [ ]:
sbas.compute_align()

## Geocoding Transform

In [ ]:
sbas.compute_geocode(1)

In [ ]:
sbas.plot_topo(quantile=[0.01, 0.99])

## Persistent Scatterers Function (PSF)

In [ ]:
# use the only selected dates for the pixels stability analysis
sbas.compute_ps()

In [ ]:
sbas.plot_psfunction(quantile=[0.01, 0.90])

In [ ]:
psmask_sbas = sbas.multilooking(sbas.psfunction(), coarsen=(1,4), wavelength=100)>0.2
topo_sbas = sbas.get_topo().interp_like(psmask_sbas, method='nearest')
landmask_sbas = psmask_sbas&(np.isfinite(topo_sbas))
landmask_sbas = utils.binary_opening(landmask_sbas, structure=np.ones((20,20)))
landmask_sbas = np.isfinite(sbas.conncomp_main(landmask_sbas))
landmask_sbas = utils.binary_closing(landmask_sbas, structure=np.ones((20,20)))
landmask_sbas = np.isfinite(psmask_sbas.where(landmask_sbas))
sbas.plot_landmask(landmask_sbas)

## SBAS Baseline

In [ ]:
baseline_pairs = sbas.sbas_pairs(days=60)
# optionally, drop dates having less then 2 pairs
#baseline_pairs = sbas.sbas_pairs_limit(baseline_pairs, limit=2, iterations=2)
# optionally, drop all pairs connected to the specified dates
#baseline_pairs = sbas.sbas_pairs_filter_dates(baseline_pairs, ['2021-01-01'])
baseline_pairs

In [ ]:
with mpl_settings({'figure.dpi': 300}):
    sbas.plot_baseline(baseline_pairs)

In [ ]:
import os
from contextlib import contextmanager

# Define a context manager for temporary matplotlib settings
@contextmanager
def mpl_settings(settings):
    import matplotlib as mpl
    original = {}
    for key in settings:
        original[key] = mpl.rcParams[key]
        mpl.rcParams[key] = settings[key]
    try:
        yield
    finally:
        for key in original:
            mpl.rcParams[key] = original[key]

# Ensure save directory exists
save_dir = 'saved_plots'
os.makedirs(save_dir, exist_ok=True)

# Define file path
baseline_plot_path = os.path.join(save_dir, 'Baseline_Pairs.png')

# Apply high-quality settings and plot
with mpl_settings({'figure.dpi': 300, 'figure.figsize': (48, 8)}):
    sbas.plot_baseline(baseline_pairs)

# Save the current figure
plt.savefig(baseline_plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Baseline plot saved to: {baseline_plot_path}")

# Optional: Show the plot
plt.show()

# Close figure to free memory
plt.close()

## SBAS Analysis

### Multi-looked Resolution for SBAS

In [ ]:
sbas.compute_interferogram_multilook(baseline_pairs, 'intf_mlook', wavelength=200, psize=32,
                                     weight=sbas.psfunction())

In [ ]:
ds_sbas = sbas.open_stack('intf_mlook')
# apply land mask
ds_sbas = ds_sbas.where(landmask_sbas)
intf_sbas = ds_sbas.phase
corr_sbas = ds_sbas.correlation
corr_sbas

In [ ]:
sbas.plot_interferograms(intf_sbas[:8], caption='SBAS Phase, [rad]')

In [ ]:
sbas.plot_correlations(corr_sbas[:8], caption='SBAS Correlation')

### Quality Check

In [ ]:
#baseline_pairs['corr'] = corr_sbas.sel(pair=baseline_pairs.pair.values).mean(['y', 'x'])
baseline_pairs['corr'] = corr_sbas.mean(['y', 'x'])
print (len(baseline_pairs))
baseline_pairs

In [ ]:
pairs_best = sbas.sbas_pairs_covering_correlation(baseline_pairs, 2)
print (len(pairs_best))
pairs_best

In [ ]:
with mpl_settings({'figure.dpi': 300}):
    sbas.plot_baseline(pairs_best)

In [ ]:
sbas.plot_baseline_correlation(baseline_pairs, pairs_best)

In [ ]:
sbas.plot_baseline_duration(baseline_pairs, column='corr', ascending=False)

In [ ]:
sbas.plot_baseline_duration(pairs_best, column='corr', ascending=False)

In [ ]:
intf_sbas = intf_sbas.sel(pair=pairs_best.pair.values)
corr_sbas = corr_sbas.sel(pair=pairs_best.pair.values)

In [ ]:
sbas.plot_interferograms(intf_sbas[:8], caption='SBAS Phase, [rad]')

In [ ]:
sbas.plot_correlations(corr_sbas[:8], caption='SBAS Correlation')

### 2D Unwrapping

In [ ]:
corr_sbas_stack = corr_sbas.mean('pair')

In [ ]:
# optionally, materialize to disk and open
corr_sbas_stack = sbas.sync_cube(corr_sbas_stack, 'corr_sbas_stack')

In [ ]:
sbas.plot_correlation_stack(corr_sbas_stack, CORRLIMIT := 0.05, caption='SBAS Stack Correlation')

In [ ]:
sbas.plot_interferograms(intf_sbas[:8].where(corr_sbas_stack>CORRLIMIT), caption='SBAS Phase, [rad]')

In [ ]:
unwrap_sbas = sbas.unwrap_snaphu(
    intf_sbas.where(corr_sbas_stack>CORRLIMIT),
    corr_sbas,
    conncomp=True
)
unwrap_sbas

In [ ]:
# optionally, materialize to disk and open
unwrap_sbas = sbas.sync_cube(unwrap_sbas, 'unwrap_sbas')

In [ ]:
sbas.plot_phases((unwrap_sbas.phase - unwrap_sbas.phase.mean(['y','x']))[:8], caption='SBAS Phase, [rad]')

In [ ]:
# select the main valid component
unwrap_sbas = sbas.conncomp_main(unwrap_sbas, 1)

In [ ]:
sbas.plot_phases((unwrap_sbas.phase - unwrap_sbas.phase.mean(['y','x']))[:8], caption='SBAS Phase, [rad]')

### Trend Correction

In [ ]:
decimator_sbas = sbas.decimator(resolution=15, grid=(1,1))
topo = decimator_sbas(sbas.get_topo())
yy, xx = xr.broadcast(topo.y, topo.x)
trend_sbas = sbas.regression(unwrap_sbas.phase,
        [topo,    topo*yy,    topo*xx,    topo*yy*xx,
         topo**2, topo**2*yy, topo**2*xx, topo**2*yy*xx,
         yy, xx, yy*xx], corr_sbas)

In [ ]:
# optionally, materialize to disk and open
trend_sbas = sbas.sync_cube(trend_sbas, 'trend_sbas')

In [ ]:
sbas.plot_phases(trend_sbas[:8], caption='SBAS Trend Phase, [rad]', quantile=[0.01, 0.99])

In [ ]:
sbas.plot_phases((unwrap_sbas.phase - trend_sbas)[:8], caption='SBAS Phase - Trend, [rad]', vmin=-np.pi, vmax=np.pi)

### Coherence-Weighted Least-Squares Solution for LOS Displacement, mm

In [ ]:
# calculate phase displacement in radians and convert to LOS displacement in millimeter
disp_sbas = sbas.los_displacement_mm(sbas.lstsq(unwrap_sbas.phase - trend_sbas, corr_sbas))

In [ ]:
# optionally, materialize to disk and open
disp_sbas = sbas.sync_cube(disp_sbas, 'disp_sbas')

In [ ]:
import os
import matplotlib.pyplot as plt

# Ensure save directory exists
save_dir = 'saved_plots'
os.makedirs(save_dir, exist_ok=True)

# Define file path
plot_path = os.path.join(save_dir, 'displaceme_tr_sbas.png')

# Plot SBAS displacements
sbas.plot_displacements(
    disp_sbas[::3],
    caption='SBAS Cumulative LOS Displacement, [mm]',
    quantile=[0.01, 0.99],
    symmetrical=True
)

# Save the current figure
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ SBAS displacement plot saved to: {plot_path}")

# Show and close
plt.show()
plt.close()

### Least-squares model for LOS Displacement, mm

In [ ]:
velocity_sbas = sbas.velocity(disp_sbas)
velocity_sbas

In [ ]:
# optionally, materialize to disk and open
velocity_sbas = sbas.sync_cube(velocity_sbas, 'velocity_sbas')

In [ ]:
fig = plt.figure(figsize=(12,4), dpi=300)

zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

ax = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
sbas.geocode(AOI.boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

ax = fig.add_subplot(1, 2, 2)
sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry)\
    .plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
AOI.boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

plt.suptitle('SBAS LOS Velocity, 2021', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Ensure save directory exists
save_dir = 'saved_plots'
os.makedirs(save_dir, exist_ok=True)

# Define file path
plot_path = os.path.join(save_dir, 'SBAS_LOS_Velocity.png')

# Plot setup
fig = plt.figure(figsize=(12, 4), dpi=300)
zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

# First subplot
ax = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
sbas.geocode(AOI.boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

# Second subplot
ax = fig.add_subplot(1, 2, 2)
sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry).plot.imshow(
    cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax
)
AOI.boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

# Super title and layout
plt.suptitle('SBAS LOS Velocity', fontsize=18)
plt.tight_layout()

# Save the current figure
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ SBAS LOS velocity plot saved to: {plot_path}")

# Show and close
plt.show()
plt.close()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Safety check: Ensure AOI and velocity_sbas have proper CRS ---
if AOI.crs is None:
    AOI.set_crs("EPSG:4326", inplace=True)

if not hasattr(velocity_sbas, 'rio') or velocity_sbas.rio.crs is None:
    velocity_sbas.rio.write_crs("EPSG:4326", inplace=True)

# --- Compute plotting limits based on robust quantiles ---
zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

# --- Create figure with two subplots ---
fig = plt.figure(figsize=(12, 4), dpi=300)

# --- First subplot: raw SBAS grid ---
ax1 = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax1)
sbas.geocode(AOI.boundary).plot(ax=ax1, color='black', linewidth=1)
ax1.set_aspect('auto')
ax1.set_title('Velocity (SBAS Grid)', fontsize=16)

# --- Second subplot: geocoded and clipped ---
ax2 = fig.add_subplot(1, 2, 2)
geo_velocity = sbas.as_geo(sbas.ra2ll(velocity_sbas))
geo_velocity.rio.write_crs("EPSG:4326", inplace=True)
geo_clipped = geo_velocity.rio.clip(AOI.geometry)

geo_clipped.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax2)
AOI.boundary.plot(ax=ax2, color='black', linewidth=1)
ax2.set_title('Velocity (Geocoded to AOI)', fontsize=16)

# --- Final plot formatting ---
plt.suptitle('SBAS LOS Velocity', fontsize=18)
plt.tight_layout()

# --- Save and show ---
save_path = '/saved_plots/SBAS_LOS_Velocity_AOI.png'
# plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Figure ready (not saved unless you uncomment save): {save_path}")




### STL model for LOS Displacement, mm

In [ ]:
plt.figure(figsize=(12, 4), dpi=300)

x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI).geometry][0]
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)
plt.plot(disp_pixel.date, disp_pixel, c='r', lw=2, label='Displacement POI')
plt.plot(stl_pixel.date, stl_pixel.trend, c='r', ls='--', lw=2, label='Trend POI')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='r', lw=1, label='Seasonal POI')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=14)
plt.title('SBAS LOS Displacement STL Decompose, 2021', fontsize=18)
plt.ylabel('Displacement, mm', fontsize=16)
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt

# --- Create a local folder named 'saved_plots' if it doesn't exist ---
save_dir = "./saved_plots"
os.makedirs(save_dir, exist_ok=True)

# --- Plot STL decomposition for Point of Interest (POI) ---
plt.figure(figsize=(12, 4), dpi=300)

# Extract coordinates of the POI
x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI).geometry][0]

# Select displacement data and perform STL decomposition
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)

# Plot displacement, trend, and seasonal components
plt.plot(disp_pixel.date, disp_pixel, c='r', lw=2, label='Displacement POI')
plt.plot(stl_pixel.date, stl_pixel.trend, c='r', ls='--', lw=2, label='Trend POI')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='r', lw=1, label='Seasonal POI')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=14)
plt.title('SBAS LOS Displacement STL Decomposition (2021)', fontsize=18)
plt.ylabel('Displacement (mm)', fontsize=16)
plt.tight_layout()

# --- Save the plot in the local 'saved_plots' directory ---
save_path = os.path.join(save_dir, "SBAS_LOS_Displacement_STL_2021.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Plot saved successfully at: {os.path.abspath(save_path)}")


## PS Analysis

Use the trend detected on possibly lower resolution unwrapped phases for higher resolution analysis.

In [ ]:
stability = sbas.psfunction()
landmask_ps = landmask_sbas.astype(int).interp_like(stability, method='nearest').astype(bool)
sbas.compute_interferogram_singlelook(pairs_best, 'intf_slook', wavelength=60,
                                        weight=stability.where(landmask_ps), phase=trend_sbas)

In [ ]:
ds_ps = sbas.open_stack('intf_slook')
intf_ps = ds_ps.phase
corr_ps = ds_ps.correlation

In [ ]:
sbas.plot_interferograms(intf_ps[:8], caption='PS Phase, [rad]')

In [ ]:
sbas.plot_correlations(corr_ps[:8], caption='PS Correlation')

### 1D Unwrapping and LOS Displacement, mm

In [ ]:
disp_ps_pairs = sbas.los_displacement_mm(sbas.unwrap1d(intf_ps))
disp_ps_pairs

In [ ]:
# optionally, materialize to disk and open
disp_ps_pairs = sbas.sync_cube(disp_ps_pairs, 'disp_ps_pairs')

### Coherence-Weighted Least-Squares Solution for LOS Displacement, mm

In [ ]:
disp_ps = sbas.lstsq(disp_ps_pairs, corr_ps)
disp_ps

In [ ]:
# optionally, materialize to disk and open
disp_ps = sbas.sync_cube(disp_ps, 'disp_ps')

In [ ]:
sbas.plot_displacements(disp_ps[::3], caption='PS Cumulative LOS Displacement, [mm]',
                        quantile=[0.01, 0.99], symmetrical=True)

In [ ]:
import os
import matplotlib.pyplot as plt

# Ensure save directory exists
save_dir = 'saved_plots'
os.makedirs(save_dir, exist_ok=True)

# Define file path
plot_path = os.path.join(save_dir, 'disp_ps.png')

# Plot PS displacements
sbas.plot_displacements(
    disp_ps[::3],
    caption='PS Cumulative LOS Displacement, [mm]',
    quantile=[0.01, 0.99],
    symmetrical=True
)

# Save the current figure
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ PS displacement plot saved to: {plot_path}")

# Show and close
plt.show()
plt.close()

### Least-squares model for LOS Displacement, mm

In [ ]:
velocity_ps = sbas.velocity(disp_ps)
velocity_ps

In [ ]:
# optionally, materialize to disk and open
velocity_ps = sbas.sync_cube(velocity_ps, 'velocity_ps')

In [ ]:
fig = plt.figure(figsize=(12,4), dpi=300)

zmin, zmax = np.nanquantile(velocity_ps, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

ax = fig.add_subplot(1, 2, 1)
velocity_ps.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
sbas.geocode(AOI.boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

ax = fig.add_subplot(1, 2, 2)
sbas.as_geo(sbas.ra2ll(velocity_ps)).rio.clip(AOI.geometry)\
    .plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
AOI.boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

plt.suptitle('PS LOS Velocity, 2021', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:


# Ensure save directory exists
save_dir = 'saved_plots'
os.makedirs(save_dir, exist_ok=True)

# Define file path
plot_path = os.path.join(save_dir, 'PS_LOS_Velocity.png')

# Plot setup
fig = plt.figure(figsize=(12, 4), dpi=300)

# Determine symmetric color limits
zmin, zmax = np.nanquantile(velocity_ps, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

# First subplot: radar coordinates
ax1 = fig.add_subplot(1, 2, 1)
velocity_ps.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax1)
sbas.geocode(AOI.boundary).plot(ax=ax1)
sbas.geocode(POI).plot(ax=ax1, marker='x', c='r', markersize=100, label='POI')
ax1.set_aspect('auto')
ax1.set_title('Velocity, mm/year', fontsize=16)

# Second subplot: geocoded and clipped to AOI
ax2 = fig.add_subplot(1, 2, 2)
geo_velocity = sbas.as_geo(sbas.ra2ll(velocity_ps)).rio.clip(AOI.geometry)
geo_velocity.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax2)
AOI.boundary.plot(ax=ax2)
POI.plot(ax=ax2, marker='x', c='r', markersize=100, label='POI')
ax2.legend(loc='upper left', fontsize=14)
ax2.set_title('Velocity, mm/year', fontsize=16)

# Add super title and layout adjustments
plt.suptitle('PS LOS Velocity', fontsize=18)
plt.tight_layout()

# Save before showing
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Figure saved to: {plot_path}")

# Show and close
plt.show()
plt.close()

### STL model for LOS Displacement, mm

In [ ]:

# Ensure save directory exists
save_dir = 'saved_plots'
os.makedirs(save_dir, exist_ok=True)

# Define file path
plot_path = os.path.join(save_dir, 'PS_LOS_STL_Decompose.png')

# Extract POI coordinates
x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI).geometry][0]

# Select nearest pixel in displacement data
disp_pixel = disp_ps.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_ps.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)

# Plotting
plt.figure(figsize=(12, 4), dpi=300)
plt.plot(disp_pixel.date, disp_pixel, c='r', lw=2, label='Displacement POI')
plt.plot(stl_pixel.date, stl_pixel.trend, c='r', ls='--', lw=2, label='Trend POI')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='r', lw=1, label='Seasonal POI')

# Formatting
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=14)
plt.title('PS LOS Displacement STL Decompose, 2022', fontsize=18)
plt.ylabel('Displacement, mm', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.grid(True)
plt.tight_layout()

# Save before showing
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Figure saved to: {plot_path}")

# Show and close
plt.show()
plt.close()

In [ ]:
x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI).geometry][0]
sbas.plot_baseline_displacement_los_mm(disp_ps_pairs.sel(y=y, x=x, method='nearest')/sbas.los_displacement_mm(1),
                                corr_ps.sel(y=y, x=x, method='nearest'),
                               caption='POI', stl=True)

In [ ]:


import os
import shutil
import sys

# Folder where all plots are saved (standard folder name)
save_dir = "./saved_plots"

# Check if the folder exists
if not os.path.exists(save_dir):
    raise FileNotFoundError(f"❌ Folder 'saved_plots' not found. Please run the plotting code first.")

# --- For Google Colab users: download as ZIP with notebook name ---
if 'google.colab' in sys.modules:
    from google.colab import files
    zip_filename = f"{notebook_name}_plots.zip"
    zip_path = f"/content/{zip_filename}"
    shutil.make_archive(f"/content/{notebook_name}_plots", 'zip', save_dir)
    files.download(zip_path)
    print(f"✅ All {notebook_name} plots zipped and downloaded successfully!")
    print(f"📦 Downloaded file: {zip_filename}")

# --- For local / Jupyter users: create ZIP file with notebook name ---
else:
    zip_filename = f"{notebook_name}_plots.zip"
    zip_path = os.path.abspath(zip_filename)
    shutil.make_archive(f"{notebook_name}_plots", 'zip', save_dir)
    print(f"✅ All {notebook_name} plots zipped successfully at: {zip_path}")
    print(f"📦 You can manually download or share this ZIP file: {zip_filename}")

print("\n📊 Package Contents:")
print(f"  • Source Directory: {save_dir}")
print(f"  • ZIP file: {zip_filename}")
print(f"  • Project: {notebook_name} Analysis Plots")

# Show contents of the saved_plots directory
if os.path.exists(save_dir):
    plot_files = [f for f in os.listdir(save_dir) if f.lower().endswith(('.png', '.jpg', '.pdf', '.svg'))]
    print(f"\n📄 Found {len(plot_files)} plot files:")
    for plot_file in sorted(plot_files)[:10]:  # Show first 10 files
        print(f"   • {plot_file}")
    if len(plot_files) > 10:
        print(f"   ... and {len(plot_files) - 10} more files")


### RMSE Error Estimation

In [ ]:
rmse_ps = sbas.rmse(disp_ps_pairs, disp_ps, corr_ps)
rmse_ps

In [ ]:
# optionally, materialize to disk and open
rmse_ps = sbas.sync_cube(rmse_ps, 'rmse_ps')

In [ ]:
sbas.plot_rmse(rmse_ps, caption='RMSE Correlation Aware, [mm]')

In [ ]:

import os
import sys
import shutil
from datetime import datetime

# ✅ Optionally mount Google Drive (uncomment if you prefer saving outputs there)
# from google.colab import drive
# drive.mount('/content/drive')
# base_dir = f'/content/drive/MyDrive/{notebook_name}_exports'

# ✅ Create main export folder with notebook name
base_dir = f'/content/{notebook_name}_exports'
os.makedirs(base_dir, exist_ok=True)
print(f"📁 Created export directory: {base_dir}")

# Function to zip and download folder with notebook naming
def zip_and_download(folder_path, zip_prefix):
    if 'google.colab' in sys.modules:
        from google.colab import files
        zip_name = f"{notebook_name}_{zip_prefix}.zip"
        zip_path = shutil.make_archive(zip_name.replace('.zip', ''), 'zip', folder_path)
        shutil.move(zip_path, f"/content/{zip_name}")
        files.download(f"/content/{zip_name}")
        print(f"✅ {notebook_name}_{zip_prefix} downloaded successfully!\n")

# ==============================
# 1️⃣ SBAS Velocity GeoTIFF
# ==============================
geotiff_dir = os.path.join(base_dir, 'sbas_velocity_geotiff')
os.makedirs(geotiff_dir, exist_ok=True)
sbas.export_geotiff(
    velocity_sbas,
    os.path.join(geotiff_dir, f'{notebook_name}_velocity_sbas'),
    caption='Exporting SBAS Velocity GeoTIFF'
)
zip_and_download(geotiff_dir, "SBAS_Velocity_GeoTIFF")

# ==============================
# 2️⃣ SBAS Displacement GeoTIFF
# ==============================
geotiff_dir = os.path.join(base_dir, 'sbas_displacement_geotiff')
os.makedirs(geotiff_dir, exist_ok=True)
sbas.export_geotiff(
    disp_sbas,
    os.path.join(geotiff_dir, f'{notebook_name}_disp_sbas'),
    caption='Exporting SBAS Displacement GeoTIFFs'
)
zip_and_download(geotiff_dir, "SBAS_Displacement_GeoTIFF")

# ==============================
# 3️⃣ PS Velocity + Displacement GeoTIFF
# ==============================
ps_geotiff_dir = os.path.join(base_dir, 'ps_geotiff')
os.makedirs(ps_geotiff_dir, exist_ok=True)
sbas.export_geotiff(
    velocity_ps,
    os.path.join(ps_geotiff_dir, f'{notebook_name}_velocity_ps'),
    caption='Exporting PS Velocity GeoTIFF'
)
sbas.export_geotiff(
    disp_ps,
    os.path.join(ps_geotiff_dir, f'{notebook_name}_disp_ps'),
    caption='Exporting PS Displacement GeoTIFF'
)
zip_and_download(ps_geotiff_dir, "PS_GeoTIFF")

# ==============================
# 4️⃣ PS Velocity + Displacement VTK
# ==============================
ps_vtk_dir = os.path.join(base_dir, 'ps_vtk')
os.makedirs(ps_vtk_dir, exist_ok=True)
sbas.export_vtk(
    velocity_ps,
    os.path.join(ps_vtk_dir, f'{notebook_name}_velocity_ps'),
    caption='Exporting PS Velocity VTK'
)
sbas.export_vtk(
    disp_ps,
    os.path.join(ps_vtk_dir, f'{notebook_name}_disp_ps'),
    caption='Exporting PS Displacement VTK'
)
zip_and_download(ps_vtk_dir, "PS_VTK")

# ==============================
# 5️⃣ SBAS Velocity + Displacement VTK
# ==============================
sbas_vtk_dir = os.path.join(base_dir, 'sbas_vtk')
os.makedirs(sbas_vtk_dir, exist_ok=True)
sbas.export_vtk(
    velocity_sbas,
    os.path.join(sbas_vtk_dir, f'{notebook_name}_velocity_sbas'),
    caption='Exporting SBAS Velocity VTK'
)
sbas.export_vtk(
    disp_sbas,
    os.path.join(sbas_vtk_dir, f'{notebook_name}_disp_sbas'),
    caption='Exporting SBAS Displacement VTK'
)
zip_and_download(sbas_vtk_dir, "SBAS_VTK")

# ==============================
# 6️⃣ Final Cleanup
# ==============================
# Optionally remove the local export folder after all downloads
# (Uncomment the next two lines if you want automatic cleanup)
# shutil.rmtree(base_dir)
# print("✅ All folders downloaded and local files cleaned up!")

print(f"🎯 All {notebook_name} exports completed successfully!")
print("=" * 50)
print(f"📁 Export directory: {base_dir}")
print("\n📦 Downloaded files (all with notebook name prefix):")
print(f"  • {notebook_name}_SBAS_Velocity_GeoTIFF.zip")
print(f"  • {notebook_name}_SBAS_Displacement_GeoTIFF.zip")
print(f"  • {notebook_name}_PS_GeoTIFF.zip")
print(f"  • {notebook_name}_PS_VTK.zip")
print(f"  • {notebook_name}_SBAS_VTK.zip")

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
!pip install pyvista simplekml pillow scipy matplotlib scikit-image rasterio

# =============================================================================
# IMPORT LIBRARIES
# =============================================================================
import os
import sys
import shutil
from datetime import datetime
import pyvista as pv
import numpy as np
import simplekml
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import zipfile

# =============================================================================
# FUNCTION DEFINITIONS
# =============================================================================

def read_vtk_data(vtk_filepath):
    """Read VTK file and extract point data."""
    try:
        mesh = pv.read(vtk_filepath)

        if 'trend' in mesh.array_names:
            velocity = mesh['trend']
        else:
            possible_fields = ['velocity', 'vel', 'deformation', 'disp', 'displacement']
            velocity_field = None
            for field in possible_fields:
                if field in mesh.array_names:
                    velocity_field = field
                    break
            if velocity_field is None:
                velocity_field = mesh.array_names[0]
            velocity = mesh[velocity_field]

        points = mesh.points
        return points, velocity, None

    except Exception as e:
        print(f"❌ Error reading VTK: {e}")
        return None, None, None

def create_velocity_kml(input_filepath, output_kml_path, resolution=1024):
    """Creates smooth velocity overlay KML from VTK file."""
    print(f"\n🎯 Processing {os.path.basename(input_filepath)} for KML Conversion")
    print("=" * 60)

    # 1. Read VTK data
    points, velocity, bounds = read_vtk_data(input_filepath)

    if points is None or velocity is None:
        print("❌ Failed to read VTK data")
        return False

    # 2. Data preparation
    valid_indices = ~np.isnan(velocity)
    if np.sum(valid_indices) == 0:
        print("❌ No valid data points")
        return False

    points = points[valid_indices]
    velocity = velocity[valid_indices]

    # Remove extreme outliers
    mean_vel = np.mean(velocity)
    std_vel = np.std(velocity)
    outlier_mask = np.abs(velocity - mean_vel) <= 3 * std_vel
    points = points[outlier_mask]
    velocity = velocity[outlier_mask]

    print(f"✓ Processing {len(velocity)} valid points")

    # 3. Calculate bounds
    lon, lat = points[:, 0], points[:, 1]
    bounds = {
        'south': float(lat.min()),
        'north': float(lat.max()),
        'west': float(lon.min()),
        'east': float(lon.max())
    }

    print(f"✓ Data bounds: {bounds}")

    # 4. Create interpolation grid
    print(f"🔄 Creating {resolution}x{resolution} interpolation grid...")
    grid_x, grid_y = np.mgrid[
        bounds['west']:bounds['east']:complex(resolution),
        bounds['south']:bounds['north']:complex(resolution)
    ]

    # 5. Interpolation
    print("🔄 Applying interpolation...")
    try:
        gridded_velocity = griddata(
            points[:, :2], velocity, (grid_x, grid_y),
            method='cubic', fill_value=np.nan
        )
        print("✓ Cubic interpolation completed")
    except:
        try:
            gridded_velocity = griddata(
                points[:, :2], velocity, (grid_x, grid_y),
                method='linear', fill_value=np.nan
            )
            print("✓ Linear interpolation completed")
        except Exception as e:
            print(f"❌ Interpolation failed: {e}")
            return False

    # 6. Apply smoothing
    sigma = resolution / 1000
    gridded_velocity = gaussian_filter(gridded_velocity, sigma=sigma)
    print(f"✓ Applied smoothing (sigma={sigma:.3f})")

    # 7. Color mapping
    vmin, vmax = np.nanpercentile(velocity, [2, 98])
    if abs(vmin) > vmax:
        vmax = abs(vmin) * 0.6

    print(f"🎨 Color range: {vmin:.3f} to {vmax:.3f} mm/year")

    # Enhanced colormap
    colors = [
        (0.00, '#8B0000'), (0.20, '#FF0000'), (0.35, '#FF4500'),
        (0.50, '#FFA500'), (0.65, '#FFFF00'), (0.80, '#ADFF2F'),
        (0.90, '#00FF00'), (1.00, '#00CED1')
    ]
    cmap = LinearSegmentedColormap.from_list('velocity_cmap', colors)

    # 8. Apply colormap
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    rgba_image = (cmap(norm(gridded_velocity)) * 255).astype(np.uint8)

    # 9. Handle transparency for NaN areas
    nan_mask = np.isnan(gridded_velocity)
    rgba_image[nan_mask] = [0, 0, 0, 0]  # Fully transparent
    rgba_image[~nan_mask, 3] = 255  # Full opacity for data areas

    # 10. Image processing
    pil_image = Image.fromarray(np.flipud(rgba_image.transpose(1, 0, 2)))
    pil_image = pil_image.filter(ImageFilter.GaussianBlur(radius=0.5))

    # 11. Save image and create KML
    img_path = os.path.splitext(output_kml_path)[0] + ".png"
    pil_image.save(img_path, format='PNG', optimize=True)
    print(f"💾 Saved overlay image: {img_path}")

    kml = simplekml.Kml(name=f"Velocity - {os.path.basename(input_filepath)}")
    ground = kml.newgroundoverlay(name='Velocity Overlay')
    ground.icon.href = os.path.basename(img_path)
    ground.latlonbox.north = bounds['north']
    ground.latlonbox.south = bounds['south']
    ground.latlonbox.east = bounds['east']
    ground.latlonbox.west = bounds['west']

    ground.description = f"""
    Velocity Overlay:
    • File: {os.path.basename(input_filepath)}
    • Data points: {len(velocity)}
    • Velocity range: {vmin:.3f} to {vmax:.3f} mm/year
    • Resolution: {resolution}x{resolution} pixels
    """

    kml.save(output_kml_path)
    print(f"✅ Success! KML created: {output_kml_path}")
    return True

# =============================================================================
# MAIN WORKFLOW: EXPORT VTK AND CONVERT TO KML (WITH NOTEBOOK NAMING)
# =============================================================================

# ✅ Create main export folder with notebook name
base_dir = f'/content/{notebook_name}_exports'
os.makedirs(base_dir, exist_ok=True)
print(f"📁 Created export directory: {base_dir}")

# ==============================
# 1️⃣ PS Velocity VTK Export
# ==============================
print("🔄 Exporting PS Velocity VTK...")
ps_vtk_dir = os.path.join(base_dir, 'ps_vtk')
os.makedirs(ps_vtk_dir, exist_ok=True)
sbas.export_vtk(
    velocity_ps,
    os.path.join(ps_vtk_dir, f'{notebook_name}_velocity_ps'),
    caption='Exporting PS Velocity VTK'
)

# ==============================
# 2️⃣ SBAS Velocity VTK Export
# ==============================
print("🔄 Exporting SBAS Velocity VTK...")
sbas_vtk_dir = os.path.join(base_dir, 'sbas_vtk')
os.makedirs(sbas_vtk_dir, exist_ok=True)
sbas.export_vtk(
    velocity_sbas,
    os.path.join(sbas_vtk_dir, f'{notebook_name}_velocity_sbas'),
    caption='Exporting SBAS Velocity VTK'
)

# ==============================
# 3️⃣ Convert VTK Files to KML
# ==============================
print("\n🎯 Converting VTK files to KML...")

# Define VTK file paths with notebook names
ps_vtk_file = os.path.join(ps_vtk_dir, f'{notebook_name}_velocity_ps.vtk')
sbas_vtk_file = os.path.join(sbas_vtk_dir, f'{notebook_name}_velocity_sbas.vtk')

# Create KML output directory with notebook name
kml_dir = f'/content/{notebook_name}_velocity_kmls'
os.makedirs(kml_dir, exist_ok=True)

# Convert PS Velocity to KML
if os.path.exists(ps_vtk_file):
    ps_kml_path = os.path.join(kml_dir, f'{notebook_name}_velocity_ps.kml')
    create_velocity_kml(ps_vtk_file, ps_kml_path, resolution=1024)
else:
    print("❌ PS VTK file not found!")

# Convert SBAS Velocity to KML
if os.path.exists(sbas_vtk_file):
    sbas_kml_path = os.path.join(kml_dir, f'{notebook_name}_velocity_sbas.kml')
    create_velocity_kml(sbas_vtk_file, sbas_kml_path, resolution=1024)
else:
    print("❌ SBAS VTK file not found!")

# ==============================
# 4️⃣ Package and Download Results - WITH NOTEBOOK NAME
# ==============================
print("\n📦 Packaging KML files for download...")

# Create final download package with notebook name
if os.path.exists(kml_dir) and os.listdir(kml_dir):
    final_zip = f'/content/{notebook_name}_Velocity_KML_Package.zip'

    # Package files with notebook naming
    with zipfile.ZipFile(final_zip, 'w') as zipf:
        for root, dirs, file_list in os.walk(kml_dir):
            for file_name in file_list:
                if file_name.endswith(('.kml', '.png')):
                    file_path = os.path.join(root, file_name)
                    arcname = os.path.basename(file_path)
                    zipf.write(file_path, arcname)
                    print(f"📄 Added: {file_name}")

    # Download with notebook name
    if 'google.colab' in sys.modules:
        from google.colab import files as colab_files
        colab_files.download(final_zip)
        print(f"\n🎉 SUCCESS! Downloaded: {notebook_name}_Velocity_KML_Package.zip")
        print("🌍 You can now view these KML files in Google Earth!")
    else:
        print(f"📁 Package created: {final_zip}")
else:
    print("❌ No KML files were created!")

# ==============================
# 5️⃣ Optional Cleanup
# ==============================
# Uncomment if you want to clean up temporary files
# shutil.rmtree(base_dir)
# shutil.rmtree(kml_dir)
# print("✅ Temporary files cleaned up!")

print(f"\n🎯 {notebook_name} WORKFLOW COMPLETED!")
print("=" * 50)
print(f"📁 VTK files exported to: {base_dir}")
print("🌍 KML files ready for Google Earth viewing!")
print(f"\n📊 What you'll receive (all named with '{notebook_name}'):")
print(f"  • {notebook_name}_velocity_ps.kml + {notebook_name}_velocity_ps.png")
print(f"  • {notebook_name}_velocity_sbas.kml + {notebook_name}_velocity_sbas.png")
print(f"  • All packaged in {notebook_name}_Velocity_KML_Package.zip")

## SBAS vs PS Comparision

In [ ]:
# crop AOI
points_sbas = sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry)
points_ps = sbas.as_geo(sbas.ra2ll(velocity_ps)).rio.clip(AOI.geometry)
points_ps = points_ps.interp_like(points_sbas, method='nearest').values.ravel()
points_sbas = points_sbas.values.ravel()
nanmask = np.isnan(points_sbas) | np.isnan(points_ps)
points_sbas = points_sbas[~nanmask]
points_ps = points_ps[~nanmask]

In [ ]:
plt.figure(figsize=(12, 4), dpi=300)
plt.scatter(points_sbas, points_ps, c='silver', alpha=1,   s=1)
plt.scatter(points_sbas, points_ps, c='b',      alpha=0.1, s=1)
plt.scatter(points_sbas, points_ps, c='g',      alpha=0.1, s=0.1)
plt.scatter(points_sbas, points_ps, c='y',      alpha=0.1, s=0.01)

# adding a 1:1 line
max_value = max(velocity_sbas.max(), velocity_ps.max())
min_value = min(velocity_sbas.min(), velocity_ps.min())
plt.plot([min_value, max_value], [min_value, max_value], 'k--')

plt.xlabel('Velocity SBAS, mm/year', fontsize=16)
plt.ylabel('Velocity PS, mm/year', fontsize=16)
plt.title('Cross-Comparison between SBAS and PS Velocity', fontsize=18)
plt.grid(True)
plt.show()

## 3D Interactive Map

In [ ]:
velocity_sbas_ll = sbas.ra2ll(velocity_sbas)
velocity_ps_ll = sbas.ra2ll(velocity_ps)

velocity_sbas_ll = sbas.as_geo(velocity_sbas_ll).rio.clip(AOI.geometry.envelope)
velocity_ps_ll = sbas.as_geo(velocity_ps_ll).rio.clip(AOI.geometry.envelope)

In [ ]:
gmap = XYZTiles().download(velocity_sbas_ll, 15)

In [ ]:
sbas.export_vtk(velocity_sbas_ll[::3,::2], 'velocity_sbas', image=gmap)
sbas.export_vtk(velocity_ps_ll[::3,::8],   'velocity_ps',   image=gmap)

In [ ]:
plotter = pv.Plotter(shape=(1, 2), notebook=True)
axes = pv.Axes(show_actor=True, actor_scale=2.0, line_width=5)

plotter.subplot(0, 0)
vtk_grid = pv.read('velocity_sbas.vtk')
mesh = vtk_grid.scale([1, 1, 0.00001]).rotate_z(135, point=axes.origin)
plotter.add_mesh(mesh.scale([1, 1, 0.999]), scalars='colors', rgb=True, ambient=0.2)
plotter.add_mesh(mesh, scalars='trend', ambient=0.2, cmap='turbo', clim=(-100,100), nan_opacity=0.1, nan_color='black')
plotter.show_axes()
plotter.add_title('SBAS LOS Velocity', font_size=32)

plotter.subplot(0, 1)
vtk_grid = pv.read('velocity_ps.vtk')
mesh = vtk_grid.scale([1, 1, 0.00001]).rotate_z(135, point=axes.origin)
plotter.add_mesh(mesh.scale([1, 1, 0.999]), scalars='colors', rgb=True, ambient=0.2)
plotter.add_mesh(mesh, scalars='trend', ambient=0.2, cmap='turbo', clim=(-100,100), nan_opacity=0.1, nan_color='black')
plotter.show_axes()
plotter.add_title('PS LOS Velocity', font_size=32)

plotter.show_axes()
plotter._on_first_render_request()
panel.panel(
    plotter.render_window, orientation_widget=plotter.renderer.axes_enabled,
    enable_keybindings=False, sizing_mode='stretch_width', min_height=600
)

## Export VTK file from Google Colab

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('velocity_sbas.vtk')
    files.download('velocity_ps.vtk')